# OCR on extracted frames

OCR engines read images, so text in videos is recognised in the frames from `02_extract_frames`, then merged across time: the same on-screen text in consecutive frames becomes one line with its first and last appearance.

Run `02_extract_frames` first and use a **GPU workspace**. Safe to re-run: finished videos are skipped.

In [ ]:
%pip install -q --no-warn-script-location easyocr

# Makes a package installed just now importable without restarting the kernel.
import importlib, site
site.addsitedir(site.getusersitepackages())
importlib.invalidate_caches()

In [ ]:
import gc, json, os

# Some libraries ask for a username, which workspaces may not have set.
os.environ.setdefault("USER", "jovyan")

from difflib import SequenceMatcher
from pathlib import Path

import easyocr
import pandas as pd
import torch
from tqdm.auto import tqdm

OUTPUT = Path(os.environ.get("SEMINAR_OUTPUT", "/corpus/derived"))
FRAMES = OUTPUT / "frames"
OCR = OUTPUT / "ocr"
SHARED_MODELS = Path(os.environ.get("SEMINAR_MODELS", "/models")) / "easyocr"

# Which extraction strategies to read. List several to compare them.
FRAME_STRATEGIES = os.environ.get("SEMINAR_FRAME_STRATEGIES", "every_2s").split(",")
LANGUAGES = ["de", "en"]
# Detections below this confidence are dropped.
MIN_CONFIDENCE = 0.4
# Text at least this similar counts as the same line.
SAME_LINE = 0.85
# Limit for a trial run, for example 3. None processes everything.
MAX_VIDEOS = int(os.environ["SEMINAR_MAX_VIDEOS"]) if os.environ.get("SEMINAR_MAX_VIDEOS") else None

## Load the model

Uses the shared model copy if available, otherwise downloads it to your home folder.

In [ ]:
assert torch.cuda.is_available(), "No GPU found. Restart your workspace and choose GPU."
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free: {free / 1024**3:.1f} of {total / 1024**3:.1f} GB")
reader = easyocr.Reader(
    LANGUAGES, gpu=True, verbose=False,
    model_storage_directory=str(SHARED_MODELS) if SHARED_MODELS.exists() else None,
)

## Run OCR

Each video's result is saved as soon as it is finished. Failures are collected and don't stop the run.

In [ ]:
def merge_lines(detections):
    """Combine repeated text across frames into lines with first and last time."""
    lines = []
    for item in detections:
        normal = item["text"].casefold().strip()
        match = next((line for line in lines if SequenceMatcher(None, normal, line["normal"]).ratio() >= SAME_LINE), None)
        if match:
            match["frames"] += 1
            match["last"] = item["time"]
            match["confidence"] = max(match["confidence"], item["confidence"])
        else:
            lines.append({"text": item["text"], "normal": normal, "first": item["time"], "last": item["time"],
                          "frames": 1, "confidence": item["confidence"]})
    return [{key: value for key, value in line.items() if key != "normal"} for line in lines]


def run_ocr(done, strategy):
    relative = done.parent.relative_to(FRAMES / strategy)
    target = (OCR / strategy / relative).with_name(relative.name + ".json")
    if target.exists():
        return
    record = json.loads(done.read_text(encoding="utf-8"))
    detections = []
    with_text = 0
    for frame in record["frames"]:
        found = [
            (text, float(confidence))
            for _, text, confidence in reader.readtext(str(done.parent / frame["file"]))
            if float(confidence) >= MIN_CONFIDENCE and text.strip()
        ]
        with_text += bool(found)
        detections += [{"text": text.strip(), "confidence": round(c, 3), "time": frame["time"], "file": frame["file"]} for text, c in found]
    lines = merge_lines(detections)
    result = {
        "id": record["id"], "source": record["source"], "strategy": strategy, "frames": len(record["frames"]),
        "frames_with_text": with_text, "text": " ".join(line["text"] for line in lines), "lines": lines, "detections": detections,
    }
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + ".partial")
    partial.write_text(json.dumps(result, ensure_ascii=False, indent=1), encoding="utf-8")
    partial.rename(target)


errors = []
for strategy in FRAME_STRATEGIES:
    jobs = sorted((FRAMES / strategy).rglob("frames.json"))
    assert jobs, f"No frames for {strategy}. Run 02_extract_frames first."
    if MAX_VIDEOS:
        jobs = jobs[:MAX_VIDEOS]
    for done in tqdm(jobs, desc=strategy):
        try:
            run_ocr(done, strategy)
        except Exception as error:
            errors.append({"strategy": strategy, "folder": str(done.parent), "error": f"{type(error).__name__}: {error}"})
print(len(errors), "errors")
pd.DataFrame(errors, columns=["strategy", "folder", "error"]).to_csv(OUTPUT / "ocr_errors.csv", index=False)
if errors and len(errors) == sum(len(list((FRAMES / s).rglob("frames.json"))) for s in FRAME_STRATEGIES):
    raise RuntimeError(f"Every video failed. First error: {errors[0]}")

## Compare strategies

Writes `ocr/summary.csv` to compare strategies. To work with the texts, run `04_build_tables` and use `texts.csv`.

In [ ]:
summary = []
for strategy in FRAME_STRATEGIES:
    rows = []
    for path in sorted(p for p in (OCR / strategy).rglob("*.json") if ".ipynb_checkpoints" not in p.parts):
        result = json.loads(path.read_text(encoding="utf-8"))
        rows.append({
            "id": result["id"], "source": result["source"], "strategy": strategy, "frames": result["frames"],
            "frames_with_text": result["frames_with_text"], "lines": len(result["lines"]),
            "chars": len(result["text"]), "text": result["text"],
        })
    table = pd.DataFrame(rows)
    summary.append({
        "strategy": strategy, "videos": len(table), "frames": int(table.frames.sum()),
        "videos_with_text": int((table.chars > 0).sum()), "mean_lines": round(table.lines.mean(), 1),
        "mean_chars": round(table.chars.mean(), 1),
    })
summary = pd.DataFrame(summary)
summary.to_csv(OCR / "summary.csv", index=False)
print(summary.to_string(index=False))
summary

## Free the GPU

Run this when you are done, so others can use the GPU.

In [ ]:
del reader
gc.collect()
torch.cuda.empty_cache()
print("GPU memory released")